# DeepPredict: 01. Exploratory Data Analysis (EDA)

This notebook conducts an in-depth exploratory analysis of the **NASA C-MAPSS Turbofan Engine Degradation Simulation Dataset (FD001)**.

### Objectives:
1. Inspect dataset structure, columns, and data types.
2. Perform data quality and integrity checks (nulls, duplicates, engine cycles).
3. Analyze engine run-to-failure cycle distributions.
4. Identify dynamic degradation sensors vs constant/non-informative sensors.
5. Compute sensor correlation with Remaining Useful Life (RUL).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import ALL_RAW_COLUMNS, SENSOR_COLUMNS, SETTING_COLUMNS, DatasetConfig
from src.data_loader import load_and_prepare_cmapss

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

## 1. Load and Inspect C-MAPSS FD001 Dataset

In [ ]:
config = DatasetConfig()
train_df, test_df, test_rul_truth, metadata = load_and_prepare_cmapss(config)

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")
print(f"Number of training engines: {metadata['train_engines']}")
print(f"Number of test engines: {metadata['test_engines']}")
train_df.head()

## 2. Engine Lifetime / Cycle Distribution Analysis

In [ ]:
max_cycles = train_df.groupby("unit_number")["time_in_cycles"].max()

plt.figure(figsize=(10, 5))
sns.histplot(max_cycles, bins=25, kde=True, color="navy")
plt.axvline(max_cycles.median(), color="crimson", linestyle="--", label=f"Median: {max_cycles.median():.0f} cycles")
plt.title("Turbofan Engine Lifetime Distribution (FD001)", fontsize=14, fontweight="bold")
plt.xlabel("Total Cycles to Failure", fontsize=12)
plt.ylabel("Engine Count", fontsize=12)
plt.legend()
plt.show()

## 3. Sensor Variance & Constant Sensor Identification
In FD001, engines operate under a single condition (Sea Level). Several sensors show zero or near-zero variance and provide no degradation signal.

In [ ]:
variances = train_df[SENSOR_COLUMNS].var()
constant_sensors = variances[variances < 1e-4].index.tolist()
dynamic_sensors = variances[variances >= 1e-4].index.tolist()

print(f"Constant / Flat Sensors ({len(constant_sensors)}): {constant_sensors}")
print(f"Informative Dynamic Sensors ({len(dynamic_sensors)}): {dynamic_sensors}")

## 4. Sensor Degradation Trajectories Across Life of Engine #1

In [ ]:
e1 = train_df[train_df["unit_number"] == 1]
key_sensors = ["s_2", "s_3", "s_4", "s_7", "s_11", "s_12"]

fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

for i, sensor in enumerate(key_sensors):
    axes[i].plot(e1["time_in_cycles"], e1[sensor], color="royalblue", lw=2)
    axes[i].set_title(f"{sensor} Trajectory", fontweight="bold")
    axes[i].set_ylabel("Raw Sensor Reading")

axes[-1].set_xlabel("Operational Cycles")
axes[-2].set_xlabel("Operational Cycles")
plt.tight_layout()
plt.show()

## 5. Sensor Correlation with RUL

In [ ]:
from src.explainability import analyze_sensor_correlations

corr = analyze_sensor_correlations(train_df, dynamic_sensors, target_col="RUL")
corr